In [14]:
import pandas as pd
import re
from pathlib import Path
from tqdm.auto import tqdm

CHUNK_DIRS = {
    "original": Path(
        "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/"
        "07_full_text_retrieval/materials_methods/combined/"
        "split_chunks_sent"
    ),
    "update_2025": Path(
        "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/"
        "07_full_text_retrieval/materials_methods/combined/update_2025/"
        "split_chunks_sent"
    ),
}

DOC_OUTPUT_FILES = {
    "original": Path(
        "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/"
        "08_IE_full_text/model_predictions/regex/"
        "arrive_doc_level_predictions.csv"
    ),
    "update_2025": Path(
        "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/"
        "08_IE_full_text/model_predictions/update_2025/regex/"
        "arrive_doc_level_predictions.csv"
    ),
}

SENTENCE_OUTPUT_FILES = {
    "original": Path(
        "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/"
        "08_IE_full_text/model_predictions/regex/"
        "arrive_sentence_matches.csv"
    ),
    "update_2025": Path(
        "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/"
        "08_IE_full_text/model_predictions/update_2025/regex/"
        "arrive_sentence_matches.csv"
    ),
}

ARRIVE_PATTERN = re.compile(
    r"\bARRIVE(?:\s+2\.0)?\b|"
    r"\bAnimal Research:\s*Reporting of In Vivo Experiments\b",
    flags=re.IGNORECASE
)

for source, chunk_dir in CHUNK_DIRS.items():

    print(f"\n=== Processing {source} ===")

    chunk_files = sorted(chunk_dir.glob("chunk_*.jsonl"))

    if not chunk_files:
        raise FileNotFoundError(
            f"No chunk files found in:\n{chunk_dir}"
        )

    all_pmids = set()
    supporting_sent_ids = {}

    sentence_output = SENTENCE_OUTPUT_FILES[source]
    sentence_output.parent.mkdir(parents=True, exist_ok=True)

    if sentence_output.exists():
        sentence_output.unlink()

    first_sentence_write = True

    for chunk_file in tqdm(
        chunk_files,
        desc=f"Scanning {source}"
    ):
        df = pd.read_json(
            chunk_file,
            lines=True
        )

        required_cols = {"PMID", "sent_txt"}
        missing_cols = required_cols - set(df.columns)

        if missing_cols:
            raise KeyError(
                f"Missing columns {missing_cols} in:\n{chunk_file}"
            )

        df["PMID"] = df["PMID"].astype(str)

        all_pmids.update(
            df["PMID"].dropna().unique()
        )

        mask = (
            df["sent_txt"]
            .fillna("")
            .str.contains(
                ARRIVE_PATTERN,
                regex=True
            )
        )

        matched = df.loc[mask].copy()

        if not matched.empty:

            # If your actual sentence ID column is called sent_id
            if "sent_id" in matched.columns:
                sent_id_col = "sent_id"

            # otherwise use another known ID column if present
            elif "sentence_id" in matched.columns:
                sent_id_col = "sentence_id"

            else:
                raise KeyError(
                    f"No sentence ID column found in {chunk_file}. "
                    f"Columns are: {df.columns.tolist()}"
                )

            # collect supporting sentence IDs per PMID
            for pmid, group in matched.groupby("PMID"):
                ids = group[sent_id_col].astype(str).tolist()

                supporting_sent_ids.setdefault(
                    pmid,
                    []
                ).extend(ids)

            matched["chunk_file"] = chunk_file.name

            matched.to_csv(
                sentence_output,
                mode="w" if first_sentence_write else "a",
                header=first_sentence_write,
                index=False
            )

            first_sentence_write = False

        del df, mask, matched

    # -------------------------
    # Create doc-level output
    # -------------------------

    arrive_df = pd.DataFrame({
        "PMID": sorted(all_pmids)
    })

    arrive_df["prediction_encoded_num"] = (
        arrive_df["PMID"]
        .isin(supporting_sent_ids.keys())
        .astype(int)
    )

    arrive_df["prediction_encoded_label"] = (
        arrive_df["prediction_encoded_num"]
        .map({
            1: "arrive-reported",
            0: "arrive-not-reported"
        })
    )

    arrive_df["supporting_sent_id"] = (
        arrive_df["PMID"]
        .map(
            lambda pmid: ",".join(
                supporting_sent_ids.get(pmid, [])
            )
        )
    )

    doc_output = DOC_OUTPUT_FILES[source]
    doc_output.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    arrive_df.to_csv(
        doc_output,
        index=False
    )

    print(f"\nSaved doc-level: {doc_output}")
    print(f"Saved sentence matches: {sentence_output}")
    print(f"Shape: {arrive_df.shape}")
    print(
        "ARRIVE reported:",
        arrive_df["prediction_encoded_num"].sum()
    )


=== Processing original ===


Scanning original:   0%|          | 0/20 [00:00<?, ?it/s]


Saved doc-level: /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/08_IE_full_text/model_predictions/regex/arrive_doc_level_predictions.csv
Saved sentence matches: /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/08_IE_full_text/model_predictions/regex/arrive_sentence_matches.csv
Shape: (371832, 4)
ARRIVE reported: 11552

=== Processing update_2025 ===


Scanning update_2025:   0%|          | 0/20 [00:00<?, ?it/s]


Saved doc-level: /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/08_IE_full_text/model_predictions/update_2025/regex/arrive_doc_level_predictions.csv
Saved sentence matches: /shares/animalwelfare.crs.uzh/Preclinical_Pipeline/08_IE_full_text/model_predictions/update_2025/regex/arrive_sentence_matches.csv
Shape: (19082, 4)
ARRIVE reported: 2448


In [15]:
pd.read_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/08_IE_full_text/model_predictions/update_2025/regex/arrive_doc_level_predictions.csv")

,PMID,prediction_encoded_num,prediction_encoded_label,supporting_sent_id
0,37392236,0,arrive-not-reported,NaN
1,37697721,1,arrive-reported,1
2,37914900,0,arrive-not-reported,NaN
3,37943365,0,arrive-not-reported,NaN
4,38059332,0,arrive-not-reported,NaN
...,...,...,...,...
19077,42422171,0,arrive-not-reported,NaN
19078,42444989,0,arrive-not-reported,NaN
19079,42488832,0,arrive-not-reported,NaN
19080,42524118,0,arrive-not-reported,NaN


In [19]:
pd.read_csv("/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/08_IE_full_text/model_predictions/update_2025/regex/sample_size_doc_level_predictions.csv")['prediction_encoded_label'].unique()

array(['sample-size-not-reported', 'sample-size-not-performed',
       'sample-size-present'], dtype=object)